<a href="https://colab.research.google.com/github/Catalina1101/INTRO-IA/blob/main/myquestions/Cata_Flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pregunta 1: Extracción de Características de Variabilidad Cardíaca (HRV)

## El Problema
En el monitoreo remoto de pacientes, los sensores de **ECG** capturan señales que deben ser procesadas para extraer indicadores de salud autonómica. La **Variabilidad de la Frecuencia Cardíaca (HRV)** es un biomarcador crítico, pero los datos crudos a menudo contienen **ruido de movimiento** y **valores atípicos** que invalidan los cálculos de dominio temporal.

## Tu Misión
Escribe una función llamada **`extract_hrv_metrics(rr_intervals_df, threshold_ms=2000)`** que realice lo siguiente:

### 1. **Limpieza**
- Filtre el `DataFrame` `rr_intervals_df` eliminando los intervalos **RR** (tiempo entre latidos) que sean:
  - **Superiores a `threshold_ms`**
  - **Inferiores a 300 ms**
- Estos se consideran **artefactos fisiológicos**.

### 2. **Transformación**
- Calcule las **diferencias sucesivas** entre intervalos RR consecutivos utilizando **operaciones vectorizadas**.

### 3. **Cálculo de Métricas**
- **RMSSD**: Raíz cuadrada de la media de las diferencias sucesivas al cuadrado
- **SDNN**: Desviación estándar de los intervalos RR normales

### 4. **Retorno**
```python
{
    "RMSSD": float,
    "SDNN": float,
    "valid_samples_count": int
}
```

## **Especificaciones Técnicas**

### **Entrada**
| Parámetro | Tipo | Descripción |
|-----------|------|-------------|
| `rr_intervals_df` | `pandas.DataFrame` | DataFrame con columna `'RR_ms'` (float) |
| `threshold_ms` | `int` | Límite superior para latidos válidos (default: 2000) |

### **Salida**
Un **diccionario** con:
- `RMSSD`: float
- `SDNN`: float  
- `valid_samples_count`: int

## **Restricciones Obligatorias**
- **Prohibido** el uso de bucles `for` para diferencias
- **Obligatorio** usar `.diff()` de pandas
- **Obligatorio** usar **numpy** para:
  - Raíz cuadrada (`np.sqrt()`)
  - Potencias (`** 2`)

In [ ]:
import pandas as pd
import numpy as np
import random

def generar_caso_de_uso_extract_hrv_metrics():
    """
    Genera un caso de prueba para la función extract_hrv_metrics.
    Incluye filtrado de artefactos y cálculo de RMSSD/SDNN.
    """
    # 1. Configuración aleatoria
    n_rows = random.randint(60, 120)
    threshold_ms = random.choice([1800, 2000, 2200])

    # 2. Generar datos: Ritmo base con variabilidad + Outliers
    rr_values = np.random.normal(loc=850, scale=120, size=n_rows)
    # Insertar ruido (valores que deben ser filtrados)
    n_noise = int(n_rows * 0.1)
    noise_indices = np.random.choice(n_rows, n_noise, replace=False)
    rr_values[noise_indices[:n_noise//2]] = 150  # Ruido muy bajo
    rr_values[noise_indices[n_noise//2:]] = 3000 # Ruido muy alto

    df = pd.DataFrame({'RR_ms': rr_values})

    # 3. Construir INPUT
    input_data = {
        'rr_intervals_df': df.copy(),
        'threshold_ms': threshold_ms
    }

    # 4. Calcular OUTPUT (Ground Truth)
    # Filtrado según la consigna: [300, threshold_ms]
    df_filtered = df[(df['RR_ms'] >= 300) & (df['RR_ms'] <= threshold_ms)].copy()

    # RMSSD: Raíz de la media de las diferencias al cuadrado
    diffs = df_filtered['RR_ms'].diff().dropna()
    rmssd = np.sqrt(np.mean(diffs**2))

    # SDNN: Desviación estándar de los intervalos válidos
    sdnn = np.std(df_filtered['RR_ms'], ddof=0)

    output_data = {
        "RMSSD": float(rmssd),
        "SDNN": float(sdnn),
        "valid_samples_count": int(len(df_filtered))
    }

    return input_data, output_data

In [ ]:
print("=" * 50)
print("=== Test 1: generar_caso_de_uso_extract_hrv_metrics ===")
try:
    input_data, output_data = generar_caso_de_uso_extract_hrv_metrics()

    # Inspeccionar INPUT
    print(f"> INPUT (keys): {list(input_data.keys())}")

    # Inspeccionar OUTPUT
    output_type = type(output_data)
    print(f"> OUTPUT (type): {output_type}")

    if output_type is dict:
        print(f"  - Dictionary keys: {list(output_data.keys())}")
        # Mostrar los valores generados para mayor claridad
        for k, v in output_data.items():
            print(f"  - {k}: {v}")
    else:
        print("  - Advertencia: Se esperaba un diccionario.")
except Exception as e:
    print(f"ERROR durante la ejecución: {e}")
print("=" * 50)

=== Test 1: generar_caso_de_uso_extract_hrv_metrics ===
> INPUT (keys): ['rr_intervals_df', 'threshold_ms']
> OUTPUT (type): <class 'dict'>
  - Dictionary keys: ['RMSSD', 'SDNN', 'valid_samples_count']
  - RMSSD: 178.25348524824736
  - SDNN: 128.55096000636559
  - valid_samples_count: 80


In [ ]:
# --- 1. FUNCIÓN SOLUCIÓN ---
def extract_hrv_metrics(rr_intervals_df, threshold_ms=2000):
    # Filtrar datos válidos
    valid_mask = (rr_intervals_df['RR_ms'] >= 300) & (rr_intervals_df['RR_ms'] <= threshold_ms)
    df_filtered = rr_intervals_df[valid_mask].copy()

    # Calcular diferencias sucesivas
    diffs = df_filtered['RR_ms'].diff().dropna()

    # Calcular métricas
    rmssd = np.sqrt(np.mean(diffs**2))
    sdnn = np.std(df_filtered['RR_ms'], ddof=0)

    return {
        "RMSSD": float(rmssd),
        "SDNN": float(sdnn),
        "valid_samples_count": int(len(df_filtered))
    }

# --- 2. AUTOMATIZACIÓN DE LA PRUEBA ---
print("=" * 60)
print("=== Evaluación Automatizada: extract_hrv_metrics ===")
try:
    # 1. Generar caso de uso
    input_data, expected_output = generar_caso_de_uso_extract_hrv_metrics()

    # 2. Ejecutar la solución inyectando el diccionario como argumentos (**kwargs)
    actual_output = extract_hrv_metrics(**input_data)

    # 3. Comprobación
    print("Ejecución exitosa de la función solución.")
    print("\nComparación de resultados:")
    for key in expected_output.keys():
        esperado = expected_output[key]
        obtenido = actual_output[key]
        # Redondeamos para evitar errores de precisión de punto flotante
        coincide = np.isclose(esperado, obtenido, rtol=1e-5) if isinstance(esperado, float) else esperado == obtenido
        marcador = True if coincide else False
        print(f"  {marcador} {key}: Esperado={esperado:.4f} | Obtenido={obtenido:.4f}")

except Exception as e:
    print(f"ERROR en la evaluación: {e}")
print("=" * 60)

=== Evaluación Automatizada: extract_hrv_metrics ===
Ejecución exitosa de la función solución.

Comparación de resultados:
  True RMSSD: Esperado=162.4248 | Obtenido=162.4248
  True SDNN: Esperado=106.1467 | Obtenido=106.1467
  True valid_samples_count: Esperado=63.0000 | Obtenido=63.0000


# **Pregunta 2: Análisis de Supervivencia para Dispositivos Médicos Implantables**

## **El Problema**
Un equipo de bioingeniería está evaluando la durabilidad de una nueva prótesis de cadera. Se dispone de datos de pacientes que incluyen el tiempo hasta la falla del dispositivo o el último seguimiento (censura).

Es necesario preparar los datos para un modelo de Cox Proportional Hazards.


## **Tu Misión**
Escribe una función llamada:

`prepare_survival_data(df, event_col, duration_col)`

que realice lo siguiente:

### 1. Codificación
Asegure que la columna `event_col` sea de tipo booleano:
- `True` si ocurrió la falla  
- `False` si fue censurado  

### 2. Escalado Sensible
- Identifique las columnas numéricas (excluyendo la duración)
- Aplique `RobustScaler` de `sklearn` para manejar valores atípicos en los datos biomédicos

### 3. Manejo de Nulos
- Impute los valores faltantes usando la mediana mediante `SimpleImputer`

### 4. Retorno
- Devuelva un solo `DataFrame` transformado que contenga:
  - Las características escaladas e imputadas
  - Las columnas de evento y duración intactas

---

## **Entrada**
- `df`: DataFrame de pandas con datos clínicos y de seguimiento  
- `event_col`: Nombre de la columna de evento (str)  
- `duration_col`: Nombre de la columna de tiempo (str)  

---

## **Salida**
- DataFrame de pandas procesado  

---

## **Restricciones**
- Use `ColumnTransformer` de `sklearn.compose`  
- Las transformaciones deben afectar **solo a las variables independientes**  
- **No debe alterar el índice original del DataFrame**

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

def generar_caso_de_uso_prepare_survival_data():
    """
    Genera un caso de prueba para la función prepare_survival_data.
    Maneja imputación por mediana y escalado robusto.
    """
    n_samples = random.randint(50, 100)

    # 1. Generar datos crudos
    data = {
        'age': np.random.uniform(40, 90, n_samples),
        'bmi': np.random.normal(27, 4, n_samples),
        'blood_pressure': np.random.normal(120, 15, n_samples),
        'event_occurred': np.random.randint(0, 2, n_samples),
        'follow_up_days': np.random.randint(10, 2000, n_samples)
    }

    df = pd.DataFrame(data)

    # 2. Inyectar complejidad: NaNs y Outliers
    df.loc[df.sample(frac=0.1).index, 'age'] = np.nan
    df.loc[0, 'bmi'] = 250.0 # Outlier extremo de IMC

    input_data = {
        'df': df.copy(),
        'event_col': 'event_occurred',
        'duration_col': 'follow_up_days'
    }

    # 3. Calcular OUTPUT (Ground Truth)
    df_expected = df.copy()
    df_expected['event_occurred'] = df_expected['event_occurred'].astype(bool)

    features = ['age', 'bmi', 'blood_pressure']

    # Imputar y escalar solo las columnas de características
    imputer = SimpleImputer(strategy='median')
    scaler = RobustScaler()

    df_expected[features] = imputer.fit_transform(df_expected[features])
    df_expected[features] = scaler.fit_transform(df_expected[features])

    output_data = df_expected

    return input_data, output_data

In [ ]:
import pandas as pd
import numpy as np

print("=" * 50)
print("=== Test 2: generar_caso_de_uso_prepare_survival_data ===")
try:
    input_data, output_data = generar_caso_de_uso_prepare_survival_data()

    # Inspeccionar INPUT
    print(f"> INPUT (keys): {list(input_data.keys())}")

    # Inspeccionar OUTPUT
    output_type = type(output_data)
    print(f"> OUTPUT (type): {output_type}")

    if issubclass(output_type, pd.DataFrame):
        print(f"  - DataFrame shape: {output_data.shape}")
        print(f"  - Columns: {list(output_data.columns)}")
    else:
        print("  - Advertencia: Se esperaba un DataFrame de Pandas.")
except Exception as e:
    print(f"ERROR durante la ejecución: {e}")
print("=" * 50)

=== Test 2: generar_caso_de_uso_prepare_survival_data ===
> INPUT (keys): ['df', 'event_col', 'duration_col']
> OUTPUT (type): <class 'pandas.core.frame.DataFrame'>
  - DataFrame shape: (88, 5)
  - Columns: ['age', 'bmi', 'blood_pressure', 'event_occurred', 'follow_up_days']


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

# --- 1. FUNCIÓN SOLUCIÓN ---
def prepare_survival_data(df, event_col, duration_col):
    df_processed = df.copy()

    # Codificar a booleano
    df_processed[event_col] = df_processed[event_col].astype(bool)

    # Identificar características (excluir evento y duración)
    features = [col for col in df_processed.columns if col not in [event_col, duration_col]]

    # Imputar y escalar
    imputer = SimpleImputer(strategy='median')
    scaler = RobustScaler()

    df_processed[features] = imputer.fit_transform(df_processed[features])
    df_processed[features] = scaler.fit_transform(df_processed[features])

    return df_processed

# --- 2. AUTOMATIZACIÓN DE LA PRUEBA ---
print("=" * 60)
print("=== Evaluación Automatizada: prepare_survival_data ===")
try:
    input_data, expected_output = generar_caso_de_uso_prepare_survival_data()
    actual_output = prepare_survival_data(**input_data)

    print(" Ejecución exitosa de la función solución.")

    # Comprobación de integridad estructural y de datos
    shape_match = expected_output.shape == actual_output.shape
    cols_match = list(expected_output.columns) == list(actual_output.columns)

    # Verificar si los DataFrames son numéricamente iguales
    pd.testing.assert_frame_equal(expected_output, actual_output, check_dtype=False)
    data_match = True

    print(f"  {True if shape_match else False} Dimensiones coinciden: {actual_output.shape}")
    print(f"  {True if cols_match else False} Nombres de columnas coinciden")
    print(f"  {True if data_match else False} Transformaciones numéricas exactas")

except AssertionError:
    print("  Los DataFrames no coinciden en sus valores exactos.")
except Exception as e:
    print(f"ERROR en la evaluación: {e}")
print("=" * 60)



=== Evaluación Automatizada: prepare_survival_data ===
 Ejecución exitosa de la función solución.
  True Dimensiones coinciden: (61, 5)
  True Nombres de columnas coinciden
  True Transformaciones numéricas exactas


In [ ]:
actual_output

,age,bmi,blood_pressure,event_occurred,follow_up_days
0,-0.028329,40.940294,0.026724,True,1693
1,0.000000,0.924052,-0.157443,False,1135
2,0.607914,-0.204705,-1.633051,False,666
3,-0.691392,0.892922,-0.733358,True,834
4,-0.657217,-1.613995,0.793514,True,453
...,...,...,...,...,...
56,-0.382991,-0.643669,-0.762796,False,643
57,-0.667734,-0.108861,-0.416896,False,1668
58,-0.046516,0.166503,-0.790341,True,414
59,0.039820,0.858677,1.170514,True,620


# **Pregunta 3: Detección de Sepsis mediante Cost-Sensitive Learning**

## **El Problema**
La detección temprana de sepsis en la UCI es un problema de clasificación extremadamente desbalanceado (pocos casos positivos pero con alto costo por falso negativo).

Un modelo estándar de Machine Learning fallaría al priorizar la exactitud global sobre la sensibilidad.



## **Tu Misión**
Escribe una función llamada:

`train_sepsis_detector(X, y, class_weight_ratio=10)`

que implemente una estrategia de aprendizaje sensible al costo:

### 1. División
- Divida los datos en entrenamiento y prueba (80/20)
- Use `stratify=y` para mantener la proporción de la clase minoritaria

### 2. Modelado
- Instancie un `RandomForestClassifier`
- Configure `class_weight` para penalizar la clase positiva según `class_weight_ratio`

### 3. Optimización
- Realice una búsqueda de hiperparámetros con `RandomizedSearchCV`
- Optimice solo:
  - `n_estimators`
  - `max_depth`

### 4. Evaluación
- Calcule la matriz de confusión sobre el conjunto de prueba

---

## **Entrada**
- `X`: Matriz de características (fisiología del paciente)  
- `y`: Vector de etiquetas (0: normal, 1: sepsis)  
- `class_weight_ratio`: Peso asignado a la clase positiva (int)  

---

## **Salida**
- Una tupla: `(mejor_modelo, matriz_de_confusión)`

---

## **Restricciones**
- Debe fijar `random_state=42` en todos los componentes necesarios  
- La métrica de optimización debe ser `'recall'`

In [ ]:
import numpy as np
import random
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

def generar_caso_de_uso_train_sepsis_detector():
    """
    Genera un caso de prueba para train_sepsis_detector.
    Dataset desbalanceado con optimización de recall.
    """
    n_samples = 250
    n_features = 6

    # 1. Generar X e y (desbalanceado 90/10)
    X = np.random.randn(n_samples, n_features)
    y = np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1])
    class_weight_ratio = random.randint(8, 12)

    input_data = {
        'X': X,
        'y': y,
        'class_weight_ratio': class_weight_ratio
    }

    # 2. Calcular OUTPUT (Ground Truth)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    rf = RandomForestClassifier(
        class_weight={0: 1, 1: class_weight_ratio},
        random_state=42
    )

    # Búsqueda de hiperparámetros reducida para el test
    param_dist = {'n_estimators': [50, 100], 'max_depth': [None, 5]}
    search = RandomizedSearchCV(
        rf, param_distributions=param_dist,
        scoring='recall', n_iter=2, random_state=42
    )
    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    output_data = (best_model, cm)

    return input_data, output_data

In [ ]:
import pandas as pd
import numpy as np

print("=" * 50)
print("=== Test 3: generar_caso_de_uso_train_sepsis_detector ===")
try:
    input_data, output_data = generar_caso_de_uso_train_sepsis_detector()

    # Inspeccionar INPUT
    print(f"> INPUT (keys): {list(input_data.keys())}")

    # Inspeccionar OUTPUT
    output_type = type(output_data)
    print(f"> OUTPUT (type): {output_type}")

    if output_type is tuple:
        for idx, item in enumerate(output_data):
            item_type = type(item)
            print(f"  - Element {idx} type: {item_type}")

            # Si es un array/matriz
            if hasattr(item, 'shape'):
                print(f"  - Element {idx} shape: {item.shape}")
            # Si es un modelo clasificador
            elif hasattr(item, 'classes_'):
                print(f"  - Element {idx} es un modelo ajustado con clases: {item.classes_}")
    else:
         print("  - Advertencia: Se esperaba una tupla.")
except Exception as e:
    print(f" ERROR durante la ejecución: {e}")
print("=" * 50)

=== Test 3: generar_caso_de_uso_train_sepsis_detector ===
> INPUT (keys): ['X', 'y', 'class_weight_ratio']
> OUTPUT (type): <class 'tuple'>
  - Element 0 type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
  - Element 0 es un modelo ajustado con clases: [0 1]
  - Element 1 type: <class 'numpy.ndarray'>
  - Element 1 shape: (2, 2)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

# --- 1. FUNCIÓN SOLUCIÓN ---
def train_sepsis_detector(X, y, class_weight_ratio):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    rf = RandomForestClassifier(
        class_weight={0: 1, 1: class_weight_ratio},
        random_state=42
    )

    param_dist = {'n_estimators': [50, 100], 'max_depth': [None, 5]}
    search = RandomizedSearchCV(
        rf, param_distributions=param_dist,
        scoring='recall', n_iter=2, random_state=42
    )
    search.fit(X_train, y_train)

    y_pred = search.best_estimator_.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    return search.best_estimator_, cm

# --- 2. AUTOMATIZACIÓN DE LA PRUEBA ---
print("=" * 60)
print("=== Evaluación Automatizada: train_sepsis_detector ===")
try:
    input_data, expected_output = generar_caso_de_uso_train_sepsis_detector()
    actual_output = train_sepsis_detector(**input_data)

    print("✅ Ejecución exitosa de la función solución.")

    expected_model, expected_cm = expected_output
    actual_model, actual_cm = actual_output

    # Comprobaciones
    params_match = expected_model.get_params() == actual_model.get_params()
    cm_match = np.array_equal(expected_cm, actual_cm)

    print(f"  {'✅' if params_match else '❌'} Hiperparámetros del mejor modelo coinciden")
    print(f"  {'✅' if cm_match else '❌'} Matriz de confusión coincide")
    if cm_match:
        print(f"      {actual_cm.flatten()} (TN, FP, FN, TP)")

except Exception as e:
    print(f"🛑 ERROR en la evaluación: {e}")
print("=" * 60)

=== Evaluación Automatizada: train_sepsis_detector ===
✅ Ejecución exitosa de la función solución.
  ✅ Hiperparámetros del mejor modelo coinciden
  ✅ Matriz de confusión coincide
      [46  0  4  0] (TN, FP, FN, TP)


# **Pregunta 4: Reducción de Dimensionalidad en Datos de Expresión Genómica**

## **El Problema**
En bioingeniería genómica, se trabaja con datasets donde el número de características (genes) es muy superior al número de muestras ($p \\gg n$).

Esto causa el "mal de la dimensionalidad". Se requiere reducir el espacio de características preservando la mayor varianza posible para identificar biomarcadores.


## **Tu Misión**
Escribe una función llamada:

`reduce_genomic_dimensions(X, variance_target=0.90)`

que automatice la reducción de dimensionalidad:

### 1. Preprocesamiento
- Estandarice las características para que tengan:
  - Media = 0  
  - Varianza = 1  

### 2. Descomposición
- Aplique Análisis de Componentes Principales (`PCA`) de `sklearn`

### 3. Selección Automática
- Determine el número mínimo de componentes necesarios para explicar al menos `variance_target` (ej. 90%) de la varianza total

### 4. Retorno
- Devuelva:
  - El modelo `PCA` ajustado  
  - Los datos proyectados en el nuevo espacio latente  

---

## **Entrada**
- `X`: Array o DataFrame con niveles de expresión de miles de genes  
- `variance_target`: Umbral de varianza explicada acumulada (float)  

---

### **Salida**
- Tupla: `(pca_model, X_reduced)`

---

### **Restricciones**
- Use `np.cumsum(pca.explained_variance_ratio_)` para identificar el punto de corte  
- El PCA debe inicializarse con `svd_solver='full'`

In [ ]:
import numpy as np
import random
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def generar_caso_de_uso_reduce_genomic_dimensions():
    """
    Genera un caso de prueba para reduce_genomic_dimensions.
    Simula datos de expresión génica con alta correlación latente.
    """
    n_samples = 30
    n_genes = 100
    variance_target = random.uniform(0.80, 0.90)

    # 1. Generar datos con estructura de varianza (no ruido puro)
    latent_space = np.random.randn(n_samples, 5)
    projection = np.random.randn(5, n_genes)
    X = np.dot(latent_space, projection) + np.random.normal(0, 0.1, (n_samples, n_genes))

    input_data = {
        'X': X,
        'variance_target': variance_target
    }

    # 2. Calcular OUTPUT (Ground Truth)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Primero ajustamos un PCA completo para ver la varianza acumulada
    pca_full = PCA(svd_solver='full')
    pca_full.fit(X_scaled)

    evr_cum = np.cumsum(pca_full.explained_variance_ratio_)
    n_components = np.argmax(evr_cum >= variance_target) + 1

    # Ajustar el PCA final con el número de componentes óptimo
    pca_final = PCA(n_components=n_components, svd_solver='full')
    X_reduced = pca_final.fit_transform(X_scaled)

    output_data = (pca_final, X_reduced)

    return input_data, output_data

In [ ]:
import pandas as pd
import numpy as np

print("=" * 50)
print("=== Test 4: generar_caso_de_uso_reduce_genomic_dimensions ===")
try:
    input_data, output_data = generar_caso_de_uso_reduce_genomic_dimensions()

    # Inspeccionar INPUT
    print(f"> INPUT (keys): {list(input_data.keys())}")

    # Inspeccionar OUTPUT
    output_type = type(output_data)
    print(f"> OUTPUT (type): {output_type}")

    if output_type is tuple:
        for idx, item in enumerate(output_data):
            item_type = type(item)
            print(f"  - Element {idx} type: {item_type}")

            # Si es la matriz transformada
            if hasattr(item, 'shape'):
                print(f"  - Element {idx} shape: {item.shape}")
            # Si es el objeto PCA
            elif hasattr(item, 'components_'):
                print(f"  - Element {idx} (PCA) ajustado con {item.n_components_} componentes.")
    else:
         print("  - Advertencia: Se esperaba una tupla.")
except Exception as e:
    print(f"ERROR durante la ejecución: {e}")
print("=" * 50)

=== Test 4: generar_caso_de_uso_reduce_genomic_dimensions ===
> INPUT (keys): ['X', 'variance_target']
> OUTPUT (type): <class 'tuple'>
  - Element 0 type: <class 'sklearn.decomposition._pca.PCA'>
  - Element 0 (PCA) ajustado con 4 componentes.
  - Element 1 type: <class 'numpy.ndarray'>
  - Element 1 shape: (30, 4)


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- 1. FUNCIÓN SOLUCIÓN ---
def reduce_genomic_dimensions(X, variance_target):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Primer pase para descubrir componentes
    pca_full = PCA(svd_solver='full')
    pca_full.fit(X_scaled)

    evr_cum = np.cumsum(pca_full.explained_variance_ratio_)
    n_components = np.argmax(evr_cum >= variance_target) + 1

    # Segundo pase definitivo
    pca_final = PCA(n_components=n_components, svd_solver='full')
    X_reduced = pca_final.fit_transform(X_scaled)

    return pca_final, X_reduced

# --- 2. AUTOMATIZACIÓN DE LA PRUEBA ---
print("=" * 60)
print("=== Evaluación Automatizada: reduce_genomic_dimensions ===")
try:
    input_data, expected_output = generar_caso_de_uso_reduce_genomic_dimensions()
    actual_output = reduce_genomic_dimensions(**input_data)

    print("Ejecución exitosa de la función solución.")

    expected_pca, expected_X = expected_output
    actual_pca, actual_X = actual_output

    # Comprobaciones
    comp_match = expected_pca.n_components_ == actual_pca.n_components_
    shape_match = expected_X.shape == actual_X.shape
    # Evaluamos si las proyecciones latentes son casi idénticas (tolerancia por float32/64)
    data_match = np.allclose(expected_X, actual_X, atol=1e-4)

    print(f"  {True if comp_match else False} Número de componentes coinciden: {actual_pca.n_components_}")
    print(f"  {True if shape_match else False} Dimensiones del array reducido coinciden: {actual_X.shape}")
    print(f"  {True if data_match else False} Transformación matricial exacta")

except Exception as e:
    print(f"ERROR en la evaluación: {e}")
print("=" * 60)

=== Evaluación Automatizada: reduce_genomic_dimensions ===
Ejecución exitosa de la función solución.
  True Número de componentes coinciden: 4
  True Dimensiones del array reducido coinciden: (30, 4)
  True Transformación matricial exacta
